# Linguistic Feature Engineering
# WELFake Fake News Detection

This notebook extracts 10 handcrafted linguistic features
from the WELFake dataset articles. These features capture
writing style patterns that are orthogonal to content-based
features like TF-IDF and GloVe embeddings.

The motivation comes from Garg and Sharma (2022) who showed
that linguistic features improved fake news detection accuracy
from 72.7% to 90.0% on the Buzzfeed dataset. Padalko et al.
(2024) confirmed on WELFake specifically that fake articles
have measurably lower readability scores and more emotionally
charged language.

The 10 features extracted are:
1.  Flesch Reading Ease Score
2.  Gunning Fog Index
3.  Word Count
4.  Average Sentence Length
5.  Punctuation Density (! and ? per character)
6.  Uppercase Word Ratio
7.  Lexical Diversity (unique words / total words)
8.  Sentiment Polarity (TextBlob, -1 to +1)
9.  Sentiment Subjectivity (TextBlob, 0 to 1)
10. Negation Count

This notebook produces no model predictions or accuracy
scores. It is purely feature extraction and analysis.

Pipeline:
1. Install and import required libraries
2. Load train, val, test splits
3. Define and test feature extraction function
4. Extract features from all three splits
5. Scale features with StandardScaler fit on train only
6. Analyse feature distributions and discriminative power
7. Compute feature correlation matrix
8. Generate violin plots and correlation heatmap
9. Save all feature arrays and scaler to Drive

## Section 1 -- Setup and Imports

TextBlob is used for sentiment analysis because it provides
reliable polarity and subjectivity scores without requiring
a GPU or large pretrained model. Textstat provides readability
metrics that implement standard formulas from linguistics research.

In [ ]:
# Install required packages not available by default in Colab
# textblob: sentiment analysis (polarity and subjectivity)
# textstat: readability metrics (Flesch, Gunning Fog)
import subprocess
subprocess.run(['pip', 'install', 'textblob', 'textstat',
                '--quiet'], check=True)

from google.colab import drive
drive.mount('/content/drive')

import os
import re
import sys
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# TextBlob provides lexicon-based sentiment scoring
# No model training required -- uses hand-annotated word lists
from textblob import TextBlob

# Textstat implements standard readability formulas
# based on syllable counts and sentence lengths
import textstat

from sklearn.preprocessing import StandardScaler

BASE          = '/content/drive/MyDrive/MSU Semester 4/Applied ML/Project'
PROCESSED_DIR = BASE + '/processed/'
SRC_DIR       = BASE + '/src/'
MODELS_DIR    = BASE + '/models/'
FIGURES_DIR   = BASE + '/figures/features/'
RESULTS_DIR   = BASE + '/results/'

os.makedirs(MODELS_DIR,  exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

sys.path.append(SRC_DIR)
from utils import set_seed, print_section

set_seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

# Define the exact names of all 10 features in extraction order
# This list is saved and loaded by the hybrid model notebook
# to ensure consistent feature ordering
FEATURE_NAMES = [
    'flesch_reading_ease',
    'gunning_fog',
    'word_count',
    'avg_sentence_length',
    'punctuation_density',
    'uppercase_ratio',
    'lexical_diversity',
    'sentiment_polarity',
    'sentiment_subjectivity',
    'negation_count'
]

NEGATION_WORDS = {
    'not', 'never', 'no', 'neither', 'nor',
    'nothing', 'nobody', 'nowhere', 'cannot', "n't"
}

print("Setup complete.")
print(f"Extracting {len(FEATURE_NAMES)} linguistic features:")
for i, name in enumerate(FEATURE_NAMES, 1):
    print(f"  {i:2d}. {name}")

## Section 2 -- Load Processed Data

All three splits produced in notebook 02 are loaded here.
Feature extraction is performed on each split independently
to avoid any information leakage between splits.

Two text sources are loaded for each split:
- `content` column (cleaned): used for readability, sentiment,
  lexical diversity, word count, and negation features.
- `text` column from the raw dataset: used for punctuation
  density and uppercase ratio only, because notebook 02
  preprocessing strips punctuation and lowercases all text,
  making those two features always zero if computed from content.

In [ ]:
print_section("Loading Processed Data")

train_df = pd.read_csv(PROCESSED_DIR + 'train_clean.csv')
val_df   = pd.read_csv(PROCESSED_DIR + 'val_clean.csv')
test_df  = pd.read_csv(PROCESSED_DIR + 'test_clean.csv')

# Fill nulls with empty string to prevent errors
# during feature extraction on rare null content rows
X_train_text = train_df['content'].fillna('').tolist()
X_val_text   = val_df['content'].fillna('').tolist()
X_test_text  = test_df['content'].fillna('').tolist()

y_train = train_df['label'].values
y_val   = val_df['label'].values
y_test  = test_df['label'].values

# Load the raw dataset to recover punctuation and capitalisation.
# Notebook 02 preprocessing strips all punctuation and lowercases
# every token, so punctuation_density and uppercase_ratio would
# always be 0.0 if computed from the cleaned content column.
# Index alignment works because the processed CSVs preserve the
# original row indices from the raw file via split_data().
RAW_PATH = PROCESSED_DIR.replace('processed/', '') + \
           'data/raw/WELFake_Dataset.csv'
raw_df = pd.read_csv(RAW_PATH)

train_raw_text = raw_df.loc[train_df.index, 'text'].fillna('').tolist()
val_raw_text   = raw_df.loc[val_df.index,   'text'].fillna('').tolist()
test_raw_text  = raw_df.loc[test_df.index,  'text'].fillna('').tolist()

print(f"Train : {len(X_train_text):,} articles")
print(f"Val   : {len(X_val_text):,} articles")
print(f"Test  : {len(X_test_text):,} articles")
print()
print("Class balance -- Train:")
print(pd.Series(y_train).value_counts(normalize=True).round(4))
print()
print(f"Raw dataset loaded from: {RAW_PATH}")
print(f"Raw train rows aligned  : {len(train_raw_text):,}")

## Section 3 -- Feature Extraction Function

A single function extracts all 10 features from one article.
It accepts two text inputs: the cleaned content used for most
features, and the original raw text used for punctuation density
and uppercase ratio so those signals are not lost to preprocessing.

Each feature captures a different stylistic dimension of the text.

Feature explanations:
- Flesch Reading Ease: high = easy to read, low = complex
- Gunning Fog: years of education needed to understand
- Word count: article length signal
- Avg sentence length: mean words per sentence (not total / sentences)
- Punctuation density: emotional/sensational writing signal (from raw text)
- Uppercase ratio: shouting/emphasis signal (from raw text)
- Lexical diversity: vocabulary richness
- Sentiment polarity: emotional tone (-1 negative, +1 positive)
- Sentiment subjectivity: opinion vs fact (0 objective, 1 subjective)
- Negation count: doubt-creation signal

In [ ]:
def extract_linguistic_features(text, raw_text=None):
    """
    Extract 10 handcrafted linguistic features from a text string.

    Features capture writing style, readability, sentiment,
    and structural patterns that differ between fake and real news.
    These signals are orthogonal to content-based TF-IDF and
    GloVe features, providing complementary information to the
    hybrid model.

    Parameters
    ----------
    text : str
        Cleaned article text (from the content column). Used for
        readability, word count, lexical diversity, sentiment,
        and negation features.
    raw_text : str or None, optional
        Original unprocessed article text. Used for punctuation
        density and uppercase ratio only. If None, falls back to
        text for those features (values will be 0 after preprocessing).

    Returns
    -------
    np.ndarray
        Array of shape (10,) containing all feature values.
        Returns zero array if text is empty or too short.
    """
    # Return zero vector for empty or very short texts
    # to avoid division by zero errors in ratio computations
    if not isinstance(text, str) or len(text.strip()) < 10:
        return np.zeros(len(FEATURE_NAMES), dtype=np.float32)

    # --- Readability Features ---

    # Flesch Reading Ease: higher = easier to read
    # Based on avg sentence length and avg syllables per word
    # Fake news tends to have inconsistent readability
    flesch = textstat.flesch_reading_ease(text)

    # Gunning Fog: estimates years of education needed
    # Professional journalism clusters around 12-14
    # Fake news varies more widely
    fog = textstat.gunning_fog(text)

    # --- Structural Features ---

    # Split into words for structural analysis
    words   = text.split()
    n_words = len(words)

    # Avoid division by zero for very short texts
    if n_words == 0:
        return np.zeros(len(FEATURE_NAMES), dtype=np.float32)

    # Split into sentences using punctuation as delimiter
    # This is a simple heuristic that works well for news text
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if s.strip()]

    # Compute mean words per sentence rather than total/count.
    # The naive total/count formula returns word_count when every
    # sentence has the same length, masking structural variation.
    words_per_sentence = [len(s.split()) for s in sentences
                          if s.strip()]
    avg_sent_len = (np.mean(words_per_sentence)
                    if words_per_sentence else 0.0)

    # Use raw_text for punctuation and uppercase because
    # preprocessing in notebook 02 strips all punctuation
    # and lowercases every token, making both features always
    # 0.0 if computed from the cleaned content column.
    source_for_style = raw_text if raw_text else text

    # Punctuation density: count ! and ? per character
    # High density signals sensational or emotional writing
    n_punct       = (source_for_style.count('!') +
                     source_for_style.count('?'))
    punct_density = n_punct / max(len(source_for_style), 1)

    # Uppercase ratio: ALL CAPS words as proportion of total
    # Fake news frequently uses caps for emphasis and alarm
    words_raw       = source_for_style.split()
    n_upper         = sum(1 for w in words_raw
                          if w.isupper() and len(w) > 1)
    uppercase_ratio = n_upper / max(len(words_raw), 1)

    # Lexical diversity: unique word types / total word tokens
    # Low diversity indicates repetitive formulaic writing
    unique_words = set(w.lower() for w in words)
    lexical_div  = len(unique_words) / n_words

    # --- Sentiment Features ---

    # TextBlob sentiment analysis uses a lexicon of annotated words
    # Polarity: -1 (very negative) to +1 (very positive)
    # Subjectivity: 0 (objective/factual) to 1 (subjective/opinion)
    # Real wire-service news targets polarity~0, subjectivity~0.1
    blob         = TextBlob(text)
    polarity     = blob.sentiment.polarity
    subjectivity = blob.sentiment.subjectivity

    # --- Negation Features ---

    # Count negation words to detect doubt-creation patterns
    # Fake news sometimes negates established facts repeatedly
    words_lower = [w.lower() for w in words]
    neg_count   = sum(1 for w in words_lower
                      if w in NEGATION_WORDS)

    return np.array([
        flesch,          # 1. Flesch reading ease
        fog,             # 2. Gunning fog index
        n_words,         # 3. Word count
        avg_sent_len,    # 4. Average sentence length
        punct_density,   # 5. Punctuation density
        uppercase_ratio, # 6. Uppercase word ratio
        lexical_div,     # 7. Lexical diversity
        polarity,        # 8. Sentiment polarity
        subjectivity,    # 9. Sentiment subjectivity
        neg_count        # 10. Negation count
    ], dtype=np.float32)


# --- Sanity check: cleaned text only (simulates missing raw) ---
sample_fake_clean = "breaking hillary clinton exposed in massive scandal you wont believe this"
sample_real_clean = "the federal reserve held interest rates steady on wednesday citing stable inflation"

# --- Sanity check: raw text with punctuation and capitalisation ---
sample_fake_raw = "BREAKING!!! Hillary Clinton EXPOSED in massive scandal!! You won't believe this!!!"
sample_real_raw = "The Federal Reserve held interest rates steady on Wednesday, citing stable inflation."

print("Sanity check -- cleaned text only (punct_density and uppercase_ratio will be 0):")
print()
print(f"Fake (cleaned): {sample_fake_clean[:60]}")
fake_feats_clean = extract_linguistic_features(sample_fake_clean)
for name, val in zip(FEATURE_NAMES, fake_feats_clean):
    print(f"  {name:<30} : {val:.4f}")

print()
print("Sanity check -- cleaned text + raw text (punct and uppercase recovered):")
print()
print(f"Fake (raw)    : {sample_fake_raw[:60]}...")
fake_feats_raw = extract_linguistic_features(
    sample_fake_clean, raw_text=sample_fake_raw)
for name, val in zip(FEATURE_NAMES, fake_feats_raw):
    print(f"  {name:<30} : {val:.4f}")

print()
print(f"Real (raw)    : {sample_real_raw[:60]}")
real_feats_raw = extract_linguistic_features(
    sample_real_clean, raw_text=sample_real_raw)
for name, val in zip(FEATURE_NAMES, real_feats_raw):
    print(f"  {name:<30} : {val:.4f}")

## Section 4 -- Extract Features from All Splits

Feature extraction is applied to train, val, and test
independently. Each article passes both its cleaned text
and its corresponding raw text to extract_linguistic_features
so that punctuation density and uppercase ratio are computed
from the unprocessed source.

The train set takes longest due to its size. A progress
counter is printed every 5,000 articles so you can monitor
that extraction is proceeding correctly.

In [ ]:
def extract_features_batch(texts, raw_texts=None, split_name=''):
    """
    Extract linguistic features from a list of text strings.

    Applies extract_linguistic_features to each text and
    stacks results into a 2D numpy array. Prints progress
    every 5,000 articles and reports total time taken.

    Parameters
    ----------
    texts : list of str
        List of cleaned article text strings.
    raw_texts : list of str or None, optional
        List of original unprocessed texts aligned by index.
        Passed through to extract_linguistic_features so that
        punctuation_density and uppercase_ratio are computed
        from the raw source rather than the cleaned text.
        If None, both features will be 0.0 for all articles.
    split_name : str
        Name of the split for progress printing.

    Returns
    -------
    np.ndarray
        Feature matrix of shape (n_articles, 10).
    """
    print(f"Extracting features from {split_name} set "
          f"({len(texts):,} articles)...")

    features = []
    start    = time.time()

    for i, text in enumerate(texts):
        # Pass the corresponding raw text so punctuation and
        # uppercase features are not zeroed out by preprocessing
        raw = raw_texts[i] if raw_texts is not None else None
        features.append(extract_linguistic_features(text, raw))

        # Print progress every 5000 articles so user
        # can confirm extraction is not hanging
        if (i + 1) % 5000 == 0:
            elapsed = time.time() - start
            pct     = (i + 1) / len(texts) * 100
            print(f"  {i+1:,} / {len(texts):,} "
                  f"({pct:.0f}%)  {elapsed:.0f}s elapsed")

    elapsed = time.time() - start
    matrix  = np.array(features, dtype=np.float32)
    print(f"  Done. Shape: {matrix.shape}  "
          f"Time: {elapsed:.1f}s")
    return matrix


print_section("Extracting Linguistic Features")

X_train_ling = extract_features_batch(
    X_train_text, raw_texts=train_raw_text, split_name='train')
X_val_ling   = extract_features_batch(
    X_val_text,   raw_texts=val_raw_text,   split_name='val')
X_test_ling  = extract_features_batch(
    X_test_text,  raw_texts=test_raw_text,  split_name='test')

print()
print("Final shapes:")
print(f"  X_train_ling : {X_train_ling.shape}")
print(f"  X_val_ling   : {X_val_ling.shape}")
print(f"  X_test_ling  : {X_test_ling.shape}")

# Check for any NaN or Inf values that could
# cause issues during model training
nan_count = np.isnan(X_train_ling).sum()
inf_count = np.isinf(X_train_ling).sum()
print(f"\nData quality check -- Train:")
print(f"  NaN values : {nan_count}")
print(f"  Inf values : {inf_count}")

# Replace any NaN or Inf with 0 as a safety measure
X_train_ling = np.nan_to_num(X_train_ling, nan=0.0, posinf=0.0)
X_val_ling   = np.nan_to_num(X_val_ling,   nan=0.0, posinf=0.0)
X_test_ling  = np.nan_to_num(X_test_ling,  nan=0.0, posinf=0.0)

## Section 5 -- Scale Features with StandardScaler

Linguistic features have very different scales:
- word_count ranges from 0 to 2000+
- sentiment_polarity ranges from -1 to +1
- punctuation_density ranges from 0 to 0.05

Without scaling, word_count would dominate the Dense layer
in the hybrid model because its raw values are hundreds of
times larger than other features.

StandardScaler transforms each feature to mean=0 and std=1,
giving all features equal influence on the model.

The scaler is fit on training data only. Val and test are
transformed using the training statistics to prevent leakage.

In [ ]:
print_section("Scaling Features with StandardScaler")

# Fit scaler on training set ONLY
# Using val or test statistics here would leak future
# information into the training process
scaler = StandardScaler()
X_train_ling_scaled = scaler.fit_transform(X_train_ling)

# Transform val and test using training set statistics
# This ensures the scaling is consistent at inference time
X_val_ling_scaled  = scaler.transform(X_val_ling)
X_test_ling_scaled = scaler.transform(X_test_ling)

print("Before scaling -- Train feature statistics:")
print(f"{'Feature':<30} {'Mean':>8} {'Std':>8} "
      f"{'Min':>8} {'Max':>8}")
print("-" * 66)
for i, name in enumerate(FEATURE_NAMES):
    col = X_train_ling[:, i]
    print(f"{name:<30} {col.mean():>8.2f} {col.std():>8.2f} "
          f"{col.min():>8.2f} {col.max():>8.2f}")

print()
print("After scaling -- Train feature statistics:")
print(f"{'Feature':<30} {'Mean':>8} {'Std':>8}")
print("-" * 48)
for i, name in enumerate(FEATURE_NAMES):
    col = X_train_ling_scaled[:, i]
    print(f"{name:<30} {col.mean():>8.4f} {col.std():>8.4f}")

# Save fitted scaler for use in hybrid model
# and Streamlit demo app
joblib.dump(scaler, MODELS_DIR + 'linguistic_scaler.joblib')
print("\nScaler saved to models/linguistic_scaler.joblib")

## Section 6 -- Feature Distribution Analysis

Comparing feature means between fake and real articles
identifies which features are most discriminative.
A large difference between fake and real means indicates
that feature provides strong classification signal.

In [ ]:
print_section("Feature Distribution Analysis")

# Create dataframes for easy groupby analysis
train_features_df = pd.DataFrame(
    X_train_ling,
    columns=FEATURE_NAMES
)
train_features_df['label'] = y_train

# Compute mean per class for each feature
fake_means = train_features_df[
    train_features_df['label'] == 0][FEATURE_NAMES].mean()
real_means = train_features_df[
    train_features_df['label'] == 1][FEATURE_NAMES].mean()

# Compute absolute difference to rank discriminative power
abs_diff = (fake_means - real_means).abs()

comparison_df = pd.DataFrame({
    'Feature'   : FEATURE_NAMES,
    'Fake Mean' : fake_means.values.round(4),
    'Real Mean' : real_means.values.round(4),
    'Abs Diff'  : abs_diff.values.round(4)
}).sort_values('Abs Diff', ascending=False)

print("Feature discriminative power (sorted by difference):")
print()
print(f"{'Feature':<30} {'Fake Mean':>10} "
      f"{'Real Mean':>10} {'Abs Diff':>10}")
print("-" * 64)
for _, row in comparison_df.iterrows():
    print(f"{row['Feature']:<30} {row['Fake Mean']:>10.4f} "
          f"{row['Real Mean']:>10.4f} {row['Abs Diff']:>10.4f}")

print()
print("Top 3 most discriminative features:")
for i, (_, row) in enumerate(
        comparison_df.head(3).iterrows(), 1):
    print(f"  {i}. {row['Feature']} "
          f"(diff={row['Abs Diff']:.4f})")

## Section 7 -- Feature Correlation Analysis

Highly correlated features (above 0.9) are redundant
and one could be removed without losing information.
This analysis confirms whether all 10 features are
necessary or whether the set could be reduced.

In [ ]:
print_section("Feature Correlation Analysis")

# Compute Pearson correlation matrix on training set
corr_matrix = pd.DataFrame(
    X_train_ling, columns=FEATURE_NAMES).corr()

# Find highly correlated pairs to check for redundancy
print("Highly correlated feature pairs (|r| > 0.7):")
found_high = False
for i in range(len(FEATURE_NAMES)):
    for j in range(i + 1, len(FEATURE_NAMES)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.7:
            print(f"  {FEATURE_NAMES[i]:<30} "
                  f"{FEATURE_NAMES[j]:<30} r={r:.4f}")
            found_high = True

if not found_high:
    print("  No pairs with |r| > 0.7 found.")
    print("  All 10 features are sufficiently independent.")

## Section 8 -- Save Feature Arrays

All three scaled feature arrays and the fitted scaler
are saved to Google Drive. The hybrid model notebook
loads these directly to build the linguistic branch
without re-extracting features.

Saving pre-extracted features avoids repeating the
slow extraction step (approximately 15-20 minutes)
every time the hybrid model is run.

In [ ]:
print_section("Saving Feature Arrays")

# Save scaled arrays as .npy files for fast loading
# .npy format preserves dtype (float32) and shape exactly
np.save(BASE + '/processed/X_train_linguistic.npy',
        X_train_ling_scaled)
np.save(BASE + '/processed/X_val_linguistic.npy',
        X_val_ling_scaled)
np.save(BASE + '/processed/X_test_linguistic.npy',
        X_test_ling_scaled)

# Save feature names list so hybrid model always uses
# the same 10 features in the same order
joblib.dump(FEATURE_NAMES,
            MODELS_DIR + 'linguistic_feature_names.joblib')

print("Files saved to Google Drive:")
print(f"  processed/X_train_linguistic.npy  "
      f"{X_train_ling_scaled.nbytes / 1024:.1f} KB")
print(f"  processed/X_val_linguistic.npy    "
      f"{X_val_ling_scaled.nbytes / 1024:.1f} KB")
print(f"  processed/X_test_linguistic.npy   "
      f"{X_test_ling_scaled.nbytes / 1024:.1f} KB")
print(f"  models/linguistic_scaler.joblib")
print(f"  models/linguistic_feature_names.joblib")

## Section 9 -- Plot: Violin Plots

Violin plots show the full distribution of each feature
for fake and real articles side by side. A wide separation
between the two violins confirms strong discriminative power.
Overlapping violins indicate weaker signal for that feature.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 10))
axes = axes.flatten()

for i, (feature, ax) in enumerate(
        zip(FEATURE_NAMES, axes)):

    # Combine data for violin plot
    fake_vals = X_train_ling[y_train == 0, i]
    real_vals = X_train_ling[y_train == 1, i]

    # Clip extreme outliers at 1st and 99th percentile
    # for cleaner visualisation
    p1  = np.percentile(
        np.concatenate([fake_vals, real_vals]), 1)
    p99 = np.percentile(
        np.concatenate([fake_vals, real_vals]), 99)
    fake_vals = np.clip(fake_vals, p1, p99)
    real_vals = np.clip(real_vals, p1, p99)

    plot_df = pd.DataFrame({
        'value': np.concatenate([fake_vals, real_vals]),
        'class': (['Fake'] * len(fake_vals) +
                  ['Real'] * len(real_vals))
    })

    sns.violinplot(
        data=plot_df,
        x='class', y='value',
        palette={'Fake': '#E76F51', 'Real': '#2A9D8F'},
        ax=ax, inner='box', cut=0
    )

    # Add mean lines for each class
    ax.axhline(y=fake_vals.mean(),
               color='#E76F51', linestyle='--',
               linewidth=1, alpha=0.7)
    ax.axhline(y=real_vals.mean(),
               color='#2A9D8F', linestyle='--',
               linewidth=1, alpha=0.7)

    ax.set_title(feature.replace('_', ' ').title(),
                 fontsize=10, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('')

fig.suptitle(
    'Linguistic Feature Distributions -- Fake vs Real\n'
    '(dashed lines show class means)',
    fontsize=16, fontweight='bold'
)
plt.tight_layout()
plt.savefig(FIGURES_DIR + '20_linguistic_violin.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/features/20_linguistic_violin.png")

### Plot: Feature Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

# Use abbreviated names for readability in heatmap
short_names = [
    'Flesch', 'Fog', 'Words', 'Sent Len',
    'Punct', 'Uppercase', 'Diversity',
    'Polarity', 'Subjectivity', 'Negation'
]

sns.heatmap(
    corr_matrix,
    annot=True, fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1, vmax=1,
    xticklabels=short_names,
    yticklabels=short_names,
    ax=ax,
    linewidths=0.5,
    square=True
)

ax.set_title(
    'Linguistic Feature Correlation Matrix',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig(FIGURES_DIR + '21_linguistic_correlation.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/features/21_linguistic_correlation.png")

## Section 10 -- Summary

In [ ]:
print_section("LINGUISTIC FEATURES SUMMARY")

print("Features extracted:")
for i, name in enumerate(FEATURE_NAMES, 1):
    row  = comparison_df[comparison_df['Feature'] == name]
    diff = row['Abs Diff'].values[0]
    print(f"  {i:2d}. {name:<30} abs_diff={diff:.4f}")

print()
print("Feature matrix shapes:")
print(f"  Train : {X_train_ling_scaled.shape}")
print(f"  Val   : {X_val_ling_scaled.shape}")
print(f"  Test  : {X_test_ling_scaled.shape}")
print()
print("Top 3 most discriminative features:")
for i, (_, row) in enumerate(
        comparison_df.head(3).iterrows(), 1):
    fake_m = row['Fake Mean']
    real_m = row['Real Mean']
    print(f"  {i}. {row['Feature']}")
    print(f"     Fake mean: {fake_m:.4f}")
    print(f"     Real mean: {real_m:.4f}")
    print(f"     Difference: {row['Abs Diff']:.4f}")
print()
print("Files saved to Google Drive:")
print("  processed/X_train_linguistic.npy")
print("  processed/X_val_linguistic.npy")
print("  processed/X_test_linguistic.npy")
print("  models/linguistic_scaler.joblib")
print("  models/linguistic_feature_names.joblib")
print("  figures/features/20_linguistic_violin.png")
print("  figures/features/21_linguistic_correlation.png")
print()
print("Next step: notebook 07 builds the 3-branch hybrid")
print("model using TF-IDF + GloVe/BiLSTM + these features.")

## Next Steps

The linguistic feature arrays are saved and ready for
use in notebook 07 (hybrid model). The hybrid model
loads X_train_linguistic.npy directly as input to the
third branch, avoiding re-extraction at training time.

The linguistic_scaler.joblib is also saved for use in
the Streamlit demo app, where it scales features extracted
from any new article the user submits for prediction.